<a href="https://colab.research.google.com/github/AhmedToto23/timetable-as-CSP/blob/main/different_way.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q ortools pandas openpyxl matplotlib

print("✅ Packages installed successfully!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.7/27.7 MB 56.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 321.1/321.1 kB 22.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobuf 6.31.1 which is incompatible.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 6.31.1 which is incompatible.
tensorflow 2.19.0 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.3, but you have protobuf 6.31.1 which is incompatible.
✅ Packages installed successfully!


In [ ]:
import os
import pandas as pd

print("🔍 CHECKING UPLOADED FILES...")

files_expected = [
    'Available_Course.csv',
    'Courses.csv',
    'Instructors.csv',
    'Rooms.csv',
    'sections_data.xlsx',
    'TimeSlots.csv'
]

files_found = []
for file in files_expected:
    if os.path.exists(file):
        files_found.append(file)
        print(f"✅ FOUND: {file}")
    else:
        print(f"❌ MISSING: {file}")

print(f"\n📊 RESULT: {len(files_found)}/6 files found")

if len(files_found) == 6:
    print("🎉 ALL FILES READY! Let's continue building...")
else:
    print("⚠️  Please upload missing files via the file explorer")

🔍 CHECKING UPLOADED FILES...
✅ FOUND: Available_Course.csv
✅ FOUND: Courses.csv
✅ FOUND: Instructors.csv
✅ FOUND: Rooms.csv
✅ FOUND: sections_data.xlsx
✅ FOUND: TimeSlots.csv

📊 RESULT: 6/6 files found
🎉 ALL FILES READY! Let's continue building...


In [ ]:
import pandas as pd
import ast

class DataInspector:
    def __init__(self):
        self.data = {}

    def load_and_inspect(self):
        """Load all files and examine their structure"""
        print("📊 STEP 2: EXAMINING DATA STRUCTURE")
        print("=" * 50)

        files = {
            'Available_Course': 'Available_Course.csv',
            'Courses': 'Courses.csv',
            'Instructors': 'Instructors.csv',
            'Rooms': 'Rooms.csv',
            'Sections': 'sections_data.xlsx',
            'TimeSlots': 'TimeSlots.csv'
        }

        for name, file in files.items():
            try:
                if file.endswith('.xlsx'):
                    df = pd.read_excel(file)
                else:
                    df = pd.read_csv(file)

                self.data[name] = df
                print(f"\n📁 {name.upper()}:")
                print(f"   Shape: {df.shape}")
                print(f"   Columns: {list(df.columns)}")
                print(f"   First 2 rows:")
                display(df.head(2))

                if name == 'Courses':
                    print(f"\n   📚 PROPER COURSE TYPE ANALYSIS:")
                    print(f"      • Total unique courses: {len(df)}")

                    df_analysis = df.copy()
                    df_analysis['Lecture'] = pd.to_numeric(df_analysis['Lecture'], errors='coerce').fillna(0).astype(int)
                    df_analysis['Lab'] = pd.to_numeric(df_analysis['Lab'], errors='coerce').fillna(0).astype(int)

                    lectures_only = len(df_analysis[(df_analysis['Lecture'] > 0) & (df_analysis['Lab'] == 0)])
                    labs_only = len(df_analysis[(df_analysis['Lecture'] == 0) & (df_analysis['Lab'] > 0)])
                    both = len(df_analysis[(df_analysis['Lecture'] > 0) & (df_analysis['Lab'] > 0)])
                    neither = len(df_analysis[(df_analysis['Lecture'] == 0) & (df_analysis['Lab'] == 0)])

                    print(f"      • Courses with lectures only: {lectures_only}")
                    print(f"      • Courses with labs only: {labs_only}")
                    print(f"      • Courses with both lectures and labs: {both}")
                    print(f"      • Courses with neither: {neither}")

                    total_calculated = lectures_only + labs_only + both + neither
                    print(f"      • Calculated total: {total_calculated}")
                    print(f"      • Data integrity: {'✅' if total_calculated == len(df) else '❌'}")


                    total_lecture_courses = len(df_analysis[df_analysis['Lecture'] > 0])
                    total_lab_courses = len(df_analysis[df_analysis['Lab'] > 0])
                    print(f"      • Total courses with lectures: {total_lecture_courses}")
                    print(f"      • Total courses with labs: {total_lab_courses}")

            except Exception as e:
                print(f"❌ Error loading {name}: {e}")

        return len(self.data) == 6

    def check_critical_columns(self):
        """Check for required columns in each dataset"""
        print("\n🔍 CHECKING REQUIRED COLUMNS")
        print("=" * 50)

        required_columns = {
            'Sections': ['Department', 'Level', 'Specialization', 'StudentCount'],
            'Instructors': ['InstructorID', 'QualifiedCourses'],
            'Rooms': ['RoomID', 'Capacity'],
            'Courses': ['CourseID', 'CourseName', 'Lecture', 'Lab'],
            'Available_Course': ['CourseID', 'Level', 'Specialization'],
            'TimeSlots': ['ID',]
        }

        missing_columns = {}
        for dataset, required in required_columns.items():
            if dataset in self.data:
                missing = [col for col in required if col not in self.data[dataset].columns]
                if missing:
                    missing_columns[dataset] = missing
                    print(f"❌ {dataset}: Missing {missing}")
                else:
                    print(f"✅ {dataset}: All required columns present")

        return missing_columns

inspector = DataInspector()

if inspector.load_and_inspect():
    missing = inspector.check_critical_columns()

    if missing:
        print(f"\n⚠️  Missing columns found. We'll need to adjust our approach.")
    else:
        print(f"\n🎉 All critical columns present! Ready for next step.")

    data_dict = inspector.data
else:
    print("❌ Failed to load data")


📊 STEP 2: EXAMINING DATA STRUCTURE

📁 AVAILABLE_COURSE:
   Shape: (45, 6)
   Columns: ['Department', 'Level', 'CourseID', 'Specialization', 'preferred_Prof', 'preferred_Assi']
   First 2 rows:


,Department,Level,CourseID,Specialization,preferred_Prof,preferred_Assi
0,CSIT,1,CSC111,Core,P35,"A10,A14,A15,A35"
1,CSIT,1,ECE111,Core,P08,A02



📁 COURSES:
   Shape: (161, 6)
   Columns: ['CourseID', 'CourseName', 'Credits', 'Lecture', 'Lab', 'Lab_Type']
   First 2 rows:


,CourseID,CourseName,Credits,Lecture,Lab,Lab_Type
0,ACM215,Ordinary Differential Equations,3,1,1,Classroom
1,ACM323,Applied Numerical Methods,3,1,1,Classroom



   📚 PROPER COURSE TYPE ANALYSIS:
      • Total unique courses: 161
      • Courses with lectures only: 39
      • Courses with labs only: 5
      • Courses with both lectures and labs: 117
      • Courses with neither: 0
      • Calculated total: 161
      • Data integrity: ✅
      • Total courses with lectures: 156
      • Total courses with labs: 122

📁 INSTRUCTORS:
   Shape: (90, 5)
   Columns: ['InstructorID', 'Name', 'Role', 'QualifiedCourses', 'Not_PreferredSlots']
   First 2 rows:


...


📁 SECTIONS:
   Shape: (36, 5)
   Columns: ['Department', 'SectionID', 'Level', 'Specialization', 'StudentCount']
   First 2 rows:


,Department,SectionID,Level,Specialization,StudentCount
0,CSIT,CSIT-1-s1,1,Core,25
1,CSIT,CSIT-1-s2,1,Core,25



📁 TIMESLOTS:
   Shape: (20, 4)
   Columns: ['ID', 'Day', 'StartTime', 'EndTime']
   First 2 rows:


,ID,Day,StartTime,EndTime
0,1,Sunday,9:00 AM,10:30 AM
1,2,Sunday,10:45 AM,12:15 PM



🔍 CHECKING REQUIRED COLUMNS
✅ Sections: All required columns present
✅ Instructors: All required columns present
✅ Rooms: All required columns present
✅ Courses: All required columns present
✅ Available_Course: All required columns present
✅ TimeSlots: All required columns present

🎉 All critical columns present! Ready for next step.


In [ ]:
import pandas as pd
import ast
import numpy as np
from collections import defaultdict

class DataNormalizer:
    def __init__(self, data_dict):
        self.data = data_dict
        self.normalized_data = {}

    def parse_list_string(self, value):
        """Safely parse string representations of lists"""
        if pd.isna(value) or value == "":
            return []
        if isinstance(value, list):
            return value

        value_str = str(value).strip()

        try:
            parsed = ast.literal_eval(value_str)
            if isinstance(parsed, list):
                return [str(item).strip() for item in parsed]
        except:
            pass

        if ',' in value_str:
            return [item.strip() for item in value_str.split(',') if item.strip()]

        return [value_str]

    def normalize_instructors(self):
        """Normalize Instructors data"""
        print("👨‍🏫 NORMALIZING INSTRUCTORS DATA...")
        df = self.data['Instructors'].copy()

        if 'QualifiedCourses' in df.columns:
            df['QualifiedCourses'] = df['QualifiedCourses'].apply(self.parse_list_string)
            print(f"   ✅ Parsed QualifiedCourses")

        if 'Not_PreferredSlots' in df.columns:
            df['Not_PreferredSlots'] = df['Not_PreferredSlots'].apply(self.parse_list_string)
            for idx, slots in enumerate(df['Not_PreferredSlots']):
                int_slots = []
                for slot in slots:
                    try:
                        int_slots.append(int(slot))
                    except:
                        pass
                df.at[idx, 'Not_PreferredSlots'] = int_slots
            print(f"   ✅ Parsed Not_PreferredSlots")

        if 'MaxLoad' not in df.columns:
            df['MaxLoad'] = df['Role'].apply(lambda x: 12 if x == 'Professor' else 18)

        self.normalized_data['Instructors'] = df
        return df

    def normalize_rooms(self):
        """Normalize Rooms data"""
        print("🏫 NORMALIZING ROOMS DATA...")
        df = self.data['Rooms'].copy()

        if 'Capacity' in df.columns:
            df['Capacity'] = pd.to_numeric(df['Capacity'], errors='coerce').fillna(30).astype(int)

        if 'Type' in df.columns:
            df['RoomType'] = df['Type'].apply(lambda x: 'Lab' if str(x).lower() == 'lab' else 'Lecture')
        elif 'RoomType' not in df.columns:

            df['RoomType'] = df['Capacity'].apply(lambda x: 'Lab' if x <= 40 else 'Lecture')

        self.normalized_data['Rooms'] = df
        return df

    def normalize_courses(self):
        """Normalize Courses data and merge with Available_Course"""
        print("📚 NORMALIZING COURSES DATA...")

        df_courses = self.data['Courses'].copy()
        df_available = self.data['Available_Course'].copy()

        column_mapping = {'Lecture': 'LectureHours', 'Lab': 'LabHours'}
        for old_col, new_col in column_mapping.items():
            if old_col in df_courses.columns and new_col not in df_courses.columns:
                df_courses[new_col] = df_courses[old_col]

        if 'LectureHours' in df_courses.columns:
            df_courses['LectureHours'] = pd.to_numeric(df_courses['LectureHours'], errors='coerce').fillna(1).astype(int)
        else:
            df_courses['LectureHours'] = 1

        if 'LabHours' in df_courses.columns:
            df_courses['LabHours'] = pd.to_numeric(df_courses['LabHours'], errors='coerce').fillna(0).astype(int)
        else:
            df_courses['LabHours'] = 0

        if 'preferred_Assi' in df_available.columns:
            df_available['preferred_Assi'] = df_available['preferred_Assi'].apply(self.parse_list_string)

        if 'preferred_Prof' in df_available.columns:
            df_available['preferred_Prof'] = df_available['preferred_Prof'].apply(self.parse_list_string)


        try:
            df_master = pd.merge(df_available, df_courses, on='CourseID', how='left')

            df_master['CourseName'] = df_master['CourseName'].fillna(df_master['CourseID'])
            print(f"   ✅ Merged courses: {len(df_master)} total course offerings")

            print(f"\n   📊 PROPER COURSE TYPE BREAKDOWN (UNIQUE COURSES):")
            unique_courses = df_courses['CourseID'].nunique()
            print(f"      • Total unique courses: {unique_courses}")

            lectures_only = len(df_courses[(df_courses['LectureHours'] > 0) & (df_courses['LabHours'] == 0)])
            labs_only = len(df_courses[(df_courses['LectureHours'] == 0) & (df_courses['LabHours'] > 0)])
            both = len(df_courses[(df_courses['LectureHours'] > 0) & (df_courses['LabHours'] > 0)])
            neither = len(df_courses[(df_courses['LectureHours'] == 0) & (df_courses['LabHours'] == 0)])

            print(f"      • Courses with lectures only: {lectures_only}")
            print(f"      • Courses with labs only: {labs_only}")
            print(f"      • Courses with both lectures and labs: {both}")
            print(f"      • Courses with neither: {neither}")

            total_calculated = lectures_only + labs_only + both + neither
            print(f"      • Calculated total: {total_calculated}")
            print(f"      • Data integrity: {'✅' if total_calculated == unique_courses else '❌'}")

            total_lecture_hours = df_master['LectureHours'].sum()
            total_lab_hours = df_master['LabHours'].sum()
            print(f"\n   📈 HOURS ANALYSIS (ALL OFFERINGS):")
            print(f"      • Total Lecture Hours: {total_lecture_hours}")
            print(f"      • Total Lab Hours: {total_lab_hours}")

        except Exception as e:
            print(f"   ⚠️  Merge failed: {e}")
            df_master = df_available

        self.normalized_data['MasterCourses'] = df_master
        return df_master

    def normalize_sections(self):
        """Normalize Sections data"""
        print("👥 NORMALIZING SECTIONS DATA...")
        df = self.data['Sections'].copy()

        if 'StudentCount' in df.columns:
            df['StudentCount'] = pd.to_numeric(df['StudentCount'], errors='coerce').fillna(0).astype(int)

        self.normalized_data['Sections'] = df
        return df

    def normalize_timeslots(self):
        """Normalize TimeSlots data"""
        print("⏰ NORMALIZING TIMESLOTS DATA...")
        df = self.data['TimeSlots'].copy()

        if 'ID' in df.columns and 'TimeSlotID' not in df.columns:
            df = df.rename(columns={'ID': 'TimeSlotID'})

        if all(col in df.columns for col in ['Day', 'StartTime', 'EndTime']):
            df['TimeSlotLabel'] = df['Day'] + ' ' + df['StartTime'] + '-' + df['EndTime']
        elif 'TimeSlotLabel' not in df.columns:
            df['TimeSlotLabel'] = [f'Time Slot {i+1}' for i in range(len(df))]

        self.normalized_data['TimeSlots'] = df
        return df

    def normalize_all(self):
        """Run all normalization steps - NO GROUP/SESSION CREATION"""
        print("\n" + "="*60)
        print("DATA NORMALIZATION ONLY (NO GROUP/SESSION CREATION)")
        print("="*60)

        self.normalize_instructors()
        self.normalize_rooms()
        self.normalize_courses()
        self.normalize_sections()
        self.normalize_timeslots()

        print(f"\n✅ NORMALIZATION COMPLETE!")
        print(f"   • Ready for group creation with ImprovedGroupCreator")
        print(f"   • Ready for session generation with ImprovedSessionGenerator")

        return self.normalized_data

print("🔄 RUNNING DATA NORMALIZATION ONLY...")
normalizer = DataNormalizer(data_dict)
normalized_data = normalizer.normalize_all()

print("\n🔍 VERIFYING NORMALIZED DATA:")
for name, df in normalized_data.items():
    print(f"   {name}: {df.shape[0]} rows, {df.shape[1]} cols")

print(f"\n🎉 READY FOR EXTERNAL GROUP CREATION AND SESSION GENERATION!")

🔄 RUNNING DATA NORMALIZATION ONLY...

DATA NORMALIZATION ONLY (NO GROUP/SESSION CREATION)
👨‍🏫 NORMALIZING INSTRUCTORS DATA...
   ✅ Parsed QualifiedCourses
   ✅ Parsed Not_PreferredSlots
🏫 NORMALIZING ROOMS DATA...
📚 NORMALIZING COURSES DATA...
   ✅ Merged courses: 45 total course offerings

   📊 PROPER COURSE TYPE BREAKDOWN (UNIQUE COURSES):
      • Total unique courses: 161
      • Courses with lectures only: 39
      • Courses with labs only: 5
      • Courses with both lectures and labs: 117
      • Courses with neither: 0
      • Calculated total: 161
      • Data integrity: ✅

   📈 HOURS ANALYSIS (ALL OFFERINGS):
      • Total Lecture Hours: 43
      • Total Lab Hours: 41
👥 NORMALIZING SECTIONS DATA...
⏰ NORMALIZING TIMESLOTS DATA...

✅ NORMALIZATION COMPLETE!
   • Ready for group creation with ImprovedGroupCreator
   • Ready for session generation with ImprovedSessionGenerator

🔍 VERIFYING NORMALIZED DATA:
   Instructors: 90 rows, 6 cols
   Rooms: 120 rows, 5 cols
   MasterCourse

In [ ]:

print("🔍 VERIFYING NORMALIZED DATA")
print("=" * 50)

for name, df in normalized_data.items():
    print(f"\n📊 {name.upper()}:")
    print(f"   Shape: {df.shape}")
    print(f"   Columns: {list(df.columns)}")

    print("   Sample data:")
    for i in range(min(2, len(df))):
        row_preview = []
        for col in df.columns[:4]:
            value = df.iloc[i][col]
            if isinstance(value, list):
                preview = f"{col}: {value[:3]}{'...' if len(value) > 3 else ''}"  # Show first 3 items
            else:
                preview = f"{col}: {str(value)[:30]}{'...' if len(str(value)) > 30 else ''}"
            row_preview.append(preview)
        print(f"      Row {i}: {', '.join(row_preview)}")

print(f"\n🎉 DATA NORMALIZATION COMPLETE!")
print("Ready for the next step!")

🔍 VERIFYING NORMALIZED DATA

📊 INSTRUCTORS:
   Shape: (90, 6)
   Columns: ['InstructorID', 'Name', 'Role', 'QualifiedCourses', 'Not_PreferredSlots', 'MaxLoad']
   Sample data:
      Row 0: InstructorID: P01, Name: Dr. Adel Fathy, Role: Professor, QualifiedCourses: ['PHY113', 'PHY123']
      Row 1: InstructorID: P02, Name: Prof. Adel Al-senn, Role: Professor, QualifiedCourses: ['LRA201']

📊 ROOMS:
   Shape: (120, 5)
   Columns: ['RoomID', 'Capacity', 'Type_of_Space', 'Type', 'RoomType']
   Sample data:
      Row 0: RoomID: B09 F1.09, Capacity: 15, Type_of_Space: Classroom, Type: Lab
      Row 1: RoomID: B09 F1.12, Capacity: 15, Type_of_Space: Classroom, Type: Lab

📊 MASTERCOURSES:
   Shape: (45, 13)
   Columns: ['Department', 'Level', 'CourseID', 'Specialization', 'preferred_Prof', 'preferred_Assi', 'CourseName', 'Credits', 'Lecture', 'Lab', 'Lab_Type', 'LectureHours', 'LabHours']
   Sample data:
      Row 0: Department: CSIT, Level: 1, CourseID: CSC111, Specialization: Core
      Row 1

In [ ]:
print("🔄 IMPROVED GROUP CREATION - HANDLING LEVELS WITHOUT CORE COURSES")
print("="*60)

class ImprovedGroupCreator:
    def __init__(self, normalized_data):
        self.data = normalized_data
        self.lecture_groups = None
        self.lab_sections = None

    def create_all_groups(self, max_lecture=75, max_lab=25):
        """Create ALL groups - only Core groups for levels that have Core courses"""
        print("👥 CREATING IMPROVED GROUPS (ONLY CORE FOR LEVELS WITH CORE COURSES)...")

        df_sections = self.data['Sections']
        df_available = self.data['MasterCourses']

        lecture_groups = []
        lab_sections = []
        group_id = 1
        lab_id = 1

        all_levels = df_available['Level'].unique()

        print(f"   • Processing levels: {list(all_levels)}")

        for level in all_levels:
            print(f"\n   🎯 PROCESSING LEVEL {level}:")


            level_students = df_sections[df_sections['Level'] == level]
            total_level_students = level_students['StudentCount'].sum()

            print(f"      Total students: {total_level_students}")


            level_has_core = len(df_available[
                (df_available['Level'] == level) &
                (df_available['Specialization'] == 'Core')
            ]) > 0

            if level_has_core and total_level_students > 0:
                n_core_groups = max(1, (total_level_students + max_lecture - 1) // max_lecture)
                base_core = total_level_students // n_core_groups
                rem_core = total_level_students % n_core_groups

                for i in range(n_core_groups):
                    core_size = base_core + (1 if i < rem_core else 0)
                    lecture_groups.append({
                        'GroupID': f'G{group_id:03d}',
                        'Level': level,
                        'Specialization': 'Core',
                        'StudentCount': core_size,
                        'TotalStudents': total_level_students,
                        'GroupType': 'Core'
                    })
                    group_id += 1
                print(f"      Core: {n_core_groups} groups (Level has Core courses)")
            elif not level_has_core:
                print(f"      Core: 0 groups (Level has NO Core courses)")

            non_core_specializations = df_available[
                (df_available['Level'] == level) &
                (df_available['Specialization'] != 'Core')
            ]['Specialization'].unique()

            for spec in non_core_specializations:
                spec_students = df_sections[
                    (df_sections['Level'] == level) &
                    (df_sections['Specialization'] == spec)
                ]
                total_spec_students = spec_students['StudentCount'].sum()

                if total_spec_students == 0:
                    continue

                n_spec_groups = max(1, (total_spec_students + max_lecture - 1) // max_lecture)
                base_spec = total_spec_students // n_spec_groups
                rem_spec = total_spec_students % n_spec_groups

                for i in range(n_spec_groups):
                    spec_size = base_spec + (1 if i < rem_spec else 0)
                    lecture_groups.append({
                        'GroupID': f'G{group_id:03d}',
                        'Level': level,
                        'Specialization': spec,
                        'StudentCount': spec_size,
                        'TotalStudents': total_spec_students,
                        'GroupType': 'Specialization'
                    })
                    group_id += 1
                print(f"      {spec}: {n_spec_groups} groups")

            level_lab_count = 0
            for _, section in level_students.iterrows():
                student_count = section['StudentCount']
                specialization = section['Specialization']

                if student_count == 0:
                    continue

                n_labs = max(1, (student_count + max_lab - 1) // max_lab)
                base_lab = student_count // n_labs
                rem_lab = student_count % n_labs

                for i in range(n_labs):
                    lab_size = base_lab + (1 if i < rem_lab else 0)
                    lab_sections.append({
                        'SectionID': f'L{lab_id:03d}',
                        'OriginalSectionID': section['SectionID'],
                        'Level': level,
                        'Specialization': specialization,
                        'StudentCount': lab_size,
                        'TotalStudents': student_count,
                        'LabType': 'Single' if n_labs == 1 else 'Multiple'
                    })
                    lab_id += 1
                    level_lab_count += 1

            print(f"      Lab: {level_lab_count} sections")

        self.lecture_groups = pd.DataFrame(lecture_groups)
        self.lab_sections = pd.DataFrame(lab_sections)

        print(f"\n" + "="*60)
        print("🎯 IMPROVED GROUP CREATION COMPLETE!")
        print("="*60)

        total_lecture_groups = len(self.lecture_groups)
        total_lab_sections = len(self.lab_sections)
        total_lecture_students = self.lecture_groups['StudentCount'].sum()
        total_lab_students = self.lab_sections['StudentCount'].sum()

        print(f"📊 FINAL SUMMARY:")
        print(f"   • Lecture Groups: {total_lecture_groups}")
        print(f"   • Lab Sections: {total_lab_sections}")
        print(f"   • Total Lecture Students: {total_lecture_students}")
        print(f"   • Total Lab Students: {total_lab_students}")

        print(f"\n📈 DETAILED BREAKDOWN BY LEVEL:")
        for level in sorted(self.lecture_groups['Level'].unique()):
            level_lecture = self.lecture_groups[self.lecture_groups['Level'] == level]
            level_lab = self.lab_sections[self.lab_sections['Level'] == level]

            core_groups = level_lecture[level_lecture['Specialization'] == 'Core']
            non_core_groups = level_lecture[level_lecture['Specialization'] != 'Core']

            print(f"\n   🎯 LEVEL {level}:")
            print(f"      Lecture: {len(level_lecture)} groups, {level_lecture['StudentCount'].sum()} students")

            if len(core_groups) > 0:
                print(f"        - Core: {len(core_groups)} groups, {core_groups['StudentCount'].sum()} students")

            for spec in non_core_groups['Specialization'].unique():
                spec_groups = non_core_groups[non_core_groups['Specialization'] == spec]
                print(f"        - {spec}: {len(spec_groups)} groups, {spec_groups['StudentCount'].sum()} students")

            print(f"      Lab: {len(level_lab)} sections, {level_lab['StudentCount'].sum()} students")

        return self.lecture_groups, self.lab_sections

print("🔄 CREATING IMPROVED GROUPS...")
improved_creator = ImprovedGroupCreator(normalized_data)
improved_lecture_groups, improved_lab_sections = improved_creator.create_all_groups()

print("\n📋 IMPROVED LECTURE GROUP EXAMPLES:")
display(improved_lecture_groups)

print("\n📋 IMPROVED LAB SECTION EXAMPLES:")
display(improved_lab_sections.head(10))

🔄 IMPROVED GROUP CREATION - HANDLING LEVELS WITHOUT CORE COURSES
🔄 CREATING IMPROVED GROUPS...
👥 CREATING IMPROVED GROUPS (ONLY CORE FOR LEVELS WITH CORE COURSES)...
   • Processing levels: [np.int64(1), np.int64(2), np.int64(3), np.int64(4)]

   🎯 PROCESSING LEVEL 1:
      Total students: 225
      Core: 3 groups (Level has Core courses)
      Lab: 9 sections

   🎯 PROCESSING LEVEL 2:
      Total students: 225
      Core: 3 groups (Level has Core courses)
      Lab: 9 sections

   🎯 PROCESSING LEVEL 3:
      Total students: 205
      Core: 3 groups (Level has Core courses)
      AID: 1 groups
      CNC: 1 groups
      CSC: 1 groups
      BIF: 1 groups
      Lab: 9 sections

   🎯 PROCESSING LEVEL 4:
      Total students: 205
      Core: 0 groups (Level has NO Core courses)
      AID: 1 groups
      CNC: 1 groups
      CSC: 1 groups
      BIF: 1 groups
      Lab: 9 sections

🎯 IMPROVED GROUP CREATION COMPLETE!
📊 FINAL SUMMARY:
   • Lecture Groups: 17
   • Lab Sections: 36
   • Total Lec

,GroupID,Level,Specialization,StudentCount,TotalStudents,GroupType
0,G001,1,Core,75,225,Core
1,G002,1,Core,75,225,Core
2,G003,1,Core,75,225,Core
3,G004,2,Core,75,225,Core
4,G005,2,Core,75,225,Core
5,G006,2,Core,75,225,Core
6,G007,3,Core,69,205,Core
7,G008,3,Core,68,205,Core
8,G009,3,Core,68,205,Core
9,G010,3,AID,75,75,Specialization



📋 IMPROVED LAB SECTION EXAMPLES:


,SectionID,OriginalSectionID,Level,Specialization,StudentCount,TotalStudents,LabType
0,L001,CSIT-1-s1,1,Core,25,25,Single
1,L002,CSIT-1-s2,1,Core,25,25,Single
2,L003,CSIT-1-s3,1,Core,25,25,Single
3,L004,CSIT-1-s4,1,Core,25,25,Single
4,L005,CSIT-1-s5,1,Core,25,25,Single
5,L006,CSIT-1-s6,1,Core,25,25,Single
6,L007,CSIT-1-s7,1,Core,25,25,Single
7,L008,CSIT-1-s8,1,Core,25,25,Single
8,L009,CSIT-1-s9,1,Core,25,25,Single
9,L010,CSIT-2-s1,2,Core,25,25,Single


In [ ]:
class ImprovedSessionGenerator:
    def __init__(self, normalized_data, lecture_groups, lab_sections):
        self.data = normalized_data
        self.lecture_groups = lecture_groups
        self.lab_sections = lab_sections
        self.sessions_df = None

    def generate_sessions(self):
        """Generate sessions using improved groups with role enforcement"""
        print("\n" + "="*60)
        print("IMPROVED SESSION GENERATION WITH ROLE ENFORCEMENT")
        print("="*60)

        df_master = self.data['MasterCourses']
        sessions = []
        session_id = 1

        print("📚 PROCESSING ALL COURSES WITH ROLE ENFORCEMENT...")

        for _, course in df_master.iterrows():
            course_id = course['CourseID']
            course_name = course.get('CourseName', str(course_id))
            level = course.get('Level', '')
            specialization = course.get('Specialization', '')


            lecture_hours = int(course.get('LectureHours', 0))
            lab_hours = int(course.get('LabHours', 0))

            if lecture_hours == 0 and lab_hours == 0:
                continue

            if specialization == 'Core':
                relevant_lecture_groups = self.lecture_groups[
                    (self.lecture_groups['Level'] == level) &
                    (self.lecture_groups['Specialization'] == 'Core')
                ]
                relevant_lab_sections = self.lab_sections[self.lab_sections['Level'] == level]

                group_type = "Core"
            else:
                relevant_lecture_groups = self.lecture_groups[
                    (self.lecture_groups['Level'] == level) &
                    (self.lecture_groups['Specialization'] == specialization)
                ]
                relevant_lab_sections = self.lab_sections[
                    (self.lab_sections['Level'] == level) &
                    (self.lab_sections['Specialization'] == specialization)
                ]
                group_type = specialization

            if len(relevant_lecture_groups) == 0 and len(relevant_lab_sections) == 0:
                print(f"   ⚠️  {course_id} (Level {level}, {specialization}): NO GROUPS FOUND - SKIPPING")
                continue

            lecture_sessions_added = 0
            for _, group in relevant_lecture_groups.iterrows():
                for hour in range(lecture_hours):
                    sessions.append({
                        'SessionID': f'S{session_id:04d}',
                        'CourseID': course_id,
                        'CourseName': course_name,
                        'ClassType': 'Lecture',
                        'GroupID': group['GroupID'],
                        'StudentCount': group['StudentCount'],
                        'Level': level,
                        'Specialization': specialization,
                        'Duration': 1,
                        'RequiredRoomType': 'Lecture',
                        'RequiredInstructorRole': 'Professor',
                        'LectureHours': lecture_hours,
                        'LabHours': lab_hours
                    })
                    session_id += 1
                    lecture_sessions_added += 1

            lab_sessions_added = 0
            for _, section in relevant_lab_sections.iterrows():
                for hour in range(lab_hours):
                    sessions.append({
                        'SessionID': f'S{session_id:04d}',
                        'CourseID': course_id,
                        'CourseName': course_name,
                        'ClassType': 'Lab',
                        'GroupID': section['SectionID'],
                        'StudentCount': section['StudentCount'],
                        'Level': level,
                        'Specialization': specialization,
                        'Duration': 1,
                        'RequiredRoomType': 'Lab',
                        'RequiredInstructorRole': 'Assistant',
                        'LectureHours': lecture_hours,
                        'LabHours': lab_hours
                    })
                    session_id += 1
                    lab_sessions_added += 1

            print(f"   {course_id} (Level {level}, {specialization}): "
                  f"{lecture_sessions_added}L + {lab_sessions_added}Lab = {lecture_sessions_added + lab_sessions_added} sessions "
                  f"[{group_type} groups]")

        self.sessions_df = pd.DataFrame(sessions)

        print(f"\n📊 IMPROVED SESSION GENERATION SUMMARY:")
        print(f"   • Total Sessions: {len(self.sessions_df):,}")
        print(f"   • Lecture Sessions: {len(self.sessions_df[self.sessions_df['ClassType'] == 'Lecture']):,} (Professors only)")
        print(f"   • Lab Sessions: {len(self.sessions_df[self.sessions_df['ClassType'] == 'Lab']):,} (Assistants only)")
        print(f"   • Unique Courses: {self.sessions_df['CourseID'].nunique()}")
        print(f"   • Unique Groups: {self.sessions_df['GroupID'].nunique()}")

        lecture_roles = self.sessions_df[self.sessions_df['ClassType'] == 'Lecture']['RequiredInstructorRole'].unique()
        lab_roles = self.sessions_df[self.sessions_df['ClassType'] == 'Lab']['RequiredInstructorRole'].unique()

        print(f"\n🔒 ROLE ENFORCEMENT VERIFICATION:")
        print(f"   • Lecture sessions require: {list(lecture_roles)}")
        print(f"   • Lab sessions require: {list(lab_roles)}")

        return self.sessions_df

In [ ]:
class GlobalTimetableSolver:
    """
    Enhanced GlobalTimetableSolver with many soft constraints and strict default weights.
    Replace your old GlobalTimetableSolver with this class.
    """

    def __init__(self, normalized_data):
        self.data = normalized_data
        self.solution = None

    def solve_global(self, all_sessions_df, time_limit_seconds=180, weights=None, enforce_5credit_hard=False,phase=1):
        """
        Solve timetable for ALL levels with role enforcement and a rich set of soft constraints.

        Parameters
        ----------
        all_sessions_df : pandas.DataFrame
            DataFrame of sessions generated by your session generator.
        time_limit_seconds : int
            CP-SAT solver time limit.
        weights : dict or None
            Optional weights for soft constraints. If None, strict defaults are used.
            Keys:
              'w_max_time', 'w_pref', 'w_5c', 'w_early', 'w_balance',
              'w_room_switch', 'w_group_gap', 'w_instr_gap', 'w_cluster', 'w_spread'
        enforce_5credit_hard : bool
            If True, make 5-credit day-exclusivity a hard constraint.
        """
        print("⚡ SOLVING: Global timetable (hard + soft constraints, strict defaults) ...")
        from ortools.sat.python import cp_model
        from collections import defaultdict
        import time
        import math
        import pandas as pd

        start_time = time.time()

        # Convert inputs into lists/dicts (expected structure)
        sessions = all_sessions_df.to_dict('records')
        instructors = self.data.get('Instructors', pd.DataFrame()).to_dict('records')
        rooms = self.data.get('Rooms', pd.DataFrame()).to_dict('records')
        timeslots = self.data.get('TimeSlots', pd.DataFrame()).to_dict('records')


        n_sessions = len(sessions)
        n_instructors = len(instructors)
        n_rooms = len(rooms)
        n_times = len(timeslots)

        print(f"   • Sessions: {n_sessions} | Times: {n_times} | Rooms: {n_rooms} | Instructors: {n_instructors}")

        model = cp_model.CpModel()
        solver = cp_model.CpSolver()

        # ==============================
        # SOFT CONSTRAINT SWITCHES
        # ==============================
        ENABLE_SOFT = True            # master switch
        ENABLE_GROUP_GAPS = False     # OFF for now
        # Weights (keep SMALL)
        W_AVOID_LATE = 1
        W_GROUP_GAP = 2
        W_SAME_ROOM = 1
        W_SAME_DAY = 1


        # === Decision variables ===
        T_vars = [model.NewIntVar(0, max(0, n_times-1), f"T_{i}") for i in range(n_sessions)]
        R_vars = [model.NewIntVar(0, max(0, n_rooms-1), f"R_{i}") for i in range(n_sessions)]
        I_vars = [model.NewIntVar(0, max(0, n_instructors-1), f"I_{i}") for i in range(n_sessions)]

        # -------------------------
        # HARD CONSTRAINTS (same logic as your existing solver, made robust)
        # -------------------------
        print("   🔒 Applying hard constraints...")

        # 1) Group: sessions of same Group cannot share same timeslot
        group_sessions = defaultdict(list)
        for i, s in enumerate(sessions):
            group_sessions[s['GroupID']].append(i)

        for gid, idxs in group_sessions.items():
            if len(idxs) > 1:
                model.AddAllDifferent([T_vars[i] for i in idxs])

        # 2) Room capacity & type
        for i, s in enumerate(sessions):
            valid_rooms = []
            for r_idx, room in enumerate(rooms):
                try:
                    cap = int(room.get('Capacity', 0))
                except:
                    cap = 0
                if cap >= int(s.get('StudentCount', 0)):
                    room_type = str(room.get('RoomType', 'Lecture')).lower()
                    required_type = str(s.get('RequiredRoomType', 'Lecture')).lower()
                    if required_type in room_type or room_type == 'general':
                        valid_rooms.append(r_idx)
            if valid_rooms:
                model.AddAllowedAssignments([R_vars[i]], [[r] for r in valid_rooms])
            else:
                print(f"      ⚠️ Session {s.get('SessionID')} has NO valid rooms (capacity/type)")

        # 3) Instructor qualification & role
        prof_qual = defaultdict(list)
        asst_qual = defaultdict(list)
        for idx, instr in enumerate(instructors):
            role = instr.get('Role', '')
            q = instr.get('QualifiedCourses', []) or []
            q = [str(x) for x in q]
            for course in q:
                if role == 'Professor':
                    prof_qual[course].append(idx)
                else:
                    asst_qual[course].append(idx)

        for i, s in enumerate(sessions):
            cid = str(s.get('CourseID', ''))
            req_role = s.get('RequiredInstructorRole', 'Professor')
            if req_role == 'Professor':
                quals = prof_qual.get(cid, [])
                if not quals:
                    quals = [idx for idx, ins in enumerate(instructors) if ins.get('Role', '') == 'Professor']
            else:
                quals = asst_qual.get(cid, [])
                if not quals:
                    quals = [idx for idx, ins in enumerate(instructors) if ins.get('Role', '') == 'Assistant']
            if quals:
                model.AddAllowedAssignments([I_vars[i]], [[q] for q in quals])
            else:
                print(f"      ⚠️ Session {s.get('SessionID')} has NO qualified {req_role}s")

        # 4) Room-time uniqueness: no two sessions use same (time,room)
        combined_tr = []
        for i in range(n_sessions):
            combined = model.NewIntVar(0, max(0, n_times * max(1, n_rooms) - 1), f"tr_{i}")
            model.Add(combined == T_vars[i] * max(1, n_rooms) + R_vars[i])
            combined_tr.append(combined)
        model.AddAllDifferent(combined_tr)

        # 5) Instructor-time uniqueness
        combined_ti = []
        for i in range(n_sessions):
            combined = model.NewIntVar(0, max(0, n_times * max(1, n_instructors) - 1), f"ti_{i}")
            model.Add(combined == T_vars[i] * max(1, n_instructors) + I_vars[i])
            combined_ti.append(combined)
        model.AddAllDifferent(combined_ti)

        # -------------------------
        # PREP: timeslot mapping helpers
        # -------------------------
        # Extract timeslot fields and build mapping from label/id -> index
        ts_ids = [str(ts.get('TimeSlotID', '')).strip() for ts in timeslots]
        ts_labels = [str(ts.get('TimeSlotLabel', '')).strip() for ts in timeslots]
        ts_days = [str(ts.get('Day', '')).strip() for ts in timeslots]

        ts_index_by_id = {ts_ids[i]: i for i in range(n_times) if ts_ids[i] != ''}
        ts_index_by_label = {ts_labels[i]: i for i in range(n_times) if ts_labels[i] != ''}

        # Map days to ints
        distinct_days = []
        for d in ts_days:
            if d != '' and d not in distinct_days:
                distinct_days.append(d)
        if not distinct_days:
            distinct_days = ['D0']
        day_to_int = {d: idx for idx, d in enumerate(distinct_days)}
        day_of_timeslot = [day_to_int.get(d, 0) for d in ts_days]
        max_day_index = max(day_of_timeslot) if day_of_timeslot else 0

        # -------------------------
        # Create b_time booleans and DayVars for each session
        # -------------------------
        print("   ✨ Building timeslot indicator booleans (b_time) and DayVars...")
        b_time = [[None for _ in range(n_times)] for _ in range(n_sessions)]
        DayVars = [model.NewIntVar(0, max(0, max_day_index), f"Day_{s}") for s in range(n_sessions)]

        for s in range(n_sessions):
            bools = []
            for t in range(n_times):
                b = model.NewBoolVar(f"b_s{s}_t{t}")
                b_time[s][t] = b
                model.Add(T_vars[s] == t).OnlyEnforceIf(b)
                model.Add(T_vars[s] != t).OnlyEnforceIf(b.Not())
                bools.append(b)
            # exactly one timeslot chosen
            model.AddExactlyOne(bools)
            # Day = sum(day_of_timeslot[t] * b_time[s][t])
            model.Add(sum(day_of_timeslot[t] * b_time[s][t] for t in range(n_times)) == DayVars[s])

        # -------------------------
        # SOFT CONSTRAINTS (strict defaults, tuneable)

        print("   ✨ Adding soft constraints ...")

        # Default strict weights
        if weights is None:
           weights = {
              "w_max_time": 40,
              "w_pref": 20,
              "w_5c": 60,
              "w_early": 5,
              "w_balance": 10,
              "w_room_switch": 12,
              "w_group_gap": 25,
              "w_cluster": 8,
           }



        # ===== 1) Instructor Not-Preferred Slots (reified) =====
        InstructorPrefViolations = []
        if ENABLE_SOFT:

          # prepare bad slots per instructor
          instructor_bad_slots = {}
          for i, instr in enumerate(instructors):
            bad = set()
            for x in instr.get("Not_PreferredSlots", []) or []:
              try:
                bad.add(int(x))
              except:
                 pass
            instructor_bad_slots[i] = bad

          for s in range(n_sessions):
           for i in range(n_instructors):
              if not instructor_bad_slots[i]:
                continue

              b_assign = model.NewBoolVar(f"assign_s{s}_i{i}")
              model.Add(I_vars[s] == i).OnlyEnforceIf(b_assign)
              model.Add(I_vars[s] != i).OnlyEnforceIf(b_assign.Not())

              for t in instructor_bad_slots[i]:
                 if t >= n_times:
                   continue
                 v = model.NewBoolVar(f"pref_violate_s{s}_i{i}_t{t}")
                 model.AddBoolAnd([b_assign, b_time[s][t]]).OnlyEnforceIf(v)
                 model.AddBoolOr([b_assign.Not(), b_time[s][t].Not(), v])
                 InstructorPrefViolations.append(v)


        # ===== 2) Five-credit day exclusivity (soft or optionally hard) =====
        FiveCreditViolations = []

        def is_five_credit(sess):
           try:
             if int(sess.get("Credits", 0)) == 5:
               return True
           except:
               pass
           try:
               return int(sess.get("LectureHours", 0)) + int(sess.get("LabHours", 0)) == 5
           except:
               return False

        five_credit_sessions = [
            i for i, s in enumerate(sessions) if is_five_credit(s)
         ]

        for i in five_credit_sessions:
            group = sessions[i]["GroupID"]
            for j in group_sessions[group]:
               if i >= j:
                 continue

               if enforce_5credit_hard:
                 # 🔒 HARD: must be on different days
                 model.Add(DayVars[i] != DayVars[j])
               else:
                  # ✨ SOFT: penalize same-day scheduling (NO enforcement!)
                  v = model.NewBoolVar(f"fiveC_same_day_{i}_{j}")

                  # v = 1  ⇔ same day (violation)
                  model.Add(DayVars[i] == DayVars[j]).OnlyEnforceIf(v)
                  model.Add(DayVars[i] != DayVars[j]).OnlyEnforceIf(v.Not())

                  # DO NOT add any constraint when v = 0
                  FiveCreditViolations.append(v)



        # ===== 3) Prefer earlier timeslots (linear penalty) =====
        EarlyPenaltyVars = []
        for s in range(n_sessions):
            p = model.NewIntVar(0, n_times - 1, f"early_{s}")
            model.Add(p == T_vars[s])
            EarlyPenaltyVars.append(p)



        # ===== 4) Instructor workload balancing =====
        InstrLoadVars = []

        for i in range(n_instructors):
            load = model.NewIntVar(0, n_sessions, f"load_{i}")
            assigns = []
            for s in range(n_sessions):
                b = model.NewBoolVar(f"load_s{s}_i{i}")
                model.Add(I_vars[s] == i).OnlyEnforceIf(b)
                model.Add(I_vars[s] != i).OnlyEnforceIf(b.Not())
                assigns.append(b)
            model.Add(load == sum(assigns))
            InstrLoadVars.append(load)

        BalancePenaltyVars = []
        for a in range(len(InstrLoadVars)):
           for b in range(a + 1, len(InstrLoadVars)):
               d = model.NewIntVar(0, n_sessions, f"diff_{a}_{b}")
               model.Add(d >= InstrLoadVars[a] - InstrLoadVars[b])
               model.Add(d >= InstrLoadVars[b] - InstrLoadVars[a])
               BalancePenaltyVars.append(d)

        # ===== 5) Room switching minimization (group adjacency) =====
        RoomSwitchVars = []

        for group, sl in group_sessions.items():
           for i in range(len(sl)):
             for j in range(i + 1, len(sl)):
                 s1, s2 = sl[i], sl[j]
                 if s1 == s2:
                   continue
                 for t in range(n_times - 1):
                     b_adj = model.NewBoolVar(f"adj_{group}_{s1}_{s2}_{t}")
                     model.AddBoolAnd([b_time[s1][t], b_time[s2][t+1]]).OnlyEnforceIf(b_adj)
                     model.AddBoolOr([b_time[s1][t].Not(), b_time[s2][t+1].Not(), b_adj])

                     eq_room = model.NewBoolVar(f"eq_{group}_{s1}_{s2}_{t}")
                     model.Add(R_vars[s1] == R_vars[s2]).OnlyEnforceIf(eq_room)
                     model.Add(R_vars[s1] != R_vars[s2]).OnlyEnforceIf(eq_room.Not())

                     v = model.NewBoolVar(f"room_switch_{group}_{s1}_{s2}_{t}")
                     model.AddBoolAnd([b_adj, eq_room.Not()]).OnlyEnforceIf(v)
                     model.AddBoolOr([b_adj.Not(), eq_room, v])
                     RoomSwitchVars.append(v)


        # ===== 6) Student group gaps (penalize isolated slots) =====
        GroupGapVars = []
        if ENABLE_SOFT and ENABLE_GROUP_GAPS:

          for group, sl in group_sessions.items():
            for t in range(1, n_times - 1):
              occ_t = model.NewBoolVar(f"occ_{group}_{t}")
              occ_p = model.NewBoolVar(f"occ_{group}_{t-1}")
              occ_n = model.NewBoolVar(f"occ_{group}_{t+1}")

              model.AddMaxEquality(occ_t, [b_time[s][t] for s in sl])
              model.AddMaxEquality(occ_p, [b_time[s][t-1] for s in sl])
              model.AddMaxEquality(occ_n, [b_time[s][t+1] for s in sl])

              # ✨ SOFT GAP INDICATOR (one-way only)
              gap = model.NewBoolVar(f"gap_{group}_{t}")
              # If gap = 1 → isolated session exists
              model.AddBoolAnd([occ_t, occ_p.Not(), occ_n.Not()]).OnlyEnforceIf(gap)
              GroupGapVars.append(gap)




        # ===== 8) Course clustering (penalize multiple sessions of same course on same day) =====
        ClusterVars = []
        from collections import defaultdict

        sessions_by_course = defaultdict(list)
        for i, s in enumerate(sessions):
           sessions_by_course[s["CourseID"]].append(i)

        for course, lst in sessions_by_course.items():
           for i in range(len(lst)):
              for j in range(i + 1, len(lst)):
                 a, b = lst[i], lst[j]
                 v = model.NewBoolVar(f"cluster_{course}_{a}_{b}")
                 model.Add(DayVars[a] == DayVars[b]).OnlyEnforceIf(v)
                 model.Add(DayVars[a] != DayVars[b]).OnlyEnforceIf(v.Not())
                 ClusterVars.append(v)





        max_time = model.NewIntVar(0, n_times - 1, "max_time")
        for s in range(n_sessions):
            model.Add(max_time >= T_vars[s])


        # Build weighted objective (strict defaults)
        if phase == 1:
           print("🟢 PHASE 1: Feasibility only")
           model.Minimize(weights["w_max_time"] * max_time)

        elif phase == 2:
            print("🟡 PHASE 2: Safe soft constraints")
            model.Minimize(
              weights["w_max_time"] * max_time +
              weights["w_early"] * sum(EarlyPenaltyVars) +
              weights["w_pref"] * sum(InstructorPrefViolations)
             )

        elif phase == 3:
            print("🔵 PHASE 3: Structural soft constraints")
            model.Minimize(
              weights["w_max_time"] * max_time +
              weights["w_early"] * sum(EarlyPenaltyVars) +
              weights["w_pref"] * sum(InstructorPrefViolations) +
              weights["w_5c"] * sum(FiveCreditViolations) +
              weights["w_group_gap"] * sum(GroupGapVars) +
              weights["w_room_switch"] * sum(RoomSwitchVars) +
              weights["w_balance"] * sum(BalancePenaltyVars) +
              weights["w_cluster"] * sum(ClusterVars)
            )




        # -------------------------
        # -------------------------
        # Solve
        # -------------------------

        solver.parameters.max_time_in_seconds = time_limit_seconds
        solver.parameters.num_search_workers = 8
        solver.parameters.log_search_progress = True     # ← FORCE LOGGING
        solver.parameters.log_to_stdout = True           # ← CRITICAL FOR COLAB
        solver.parameters.cp_model_probing_level = 2




        print(f"   ⏱ Solving (time limit = {time_limit_seconds}s) ...")
        status = solver.Solve(model)
        elapsed = time.time() - start_time

        status_names = {
            cp_model.OPTIMAL: 'OPTIMAL',
            cp_model.FEASIBLE: 'FEASIBLE',
            cp_model.INFEASIBLE: 'INFEASIBLE',
            cp_model.UNKNOWN: 'UNKNOWN'
        }
        status_name = status_names.get(status, 'UNKNOWN')
        print(f"   🎯 Solver status: {status_name} (elapsed {elapsed:.1f}s)")

        if status in (cp_model.OPTIMAL, cp_model.FEASIBLE):
            self.solution = self._extract_global_solution(sessions, solver, T_vars, R_vars, I_vars, rooms, instructors, timeslots)
            self._validate_global_solution_with_roles(self.solution, instructors)
            return self.solution
        else:
            print("   ❌ No solution found.")
            return None

    def _extract_global_solution(self, sessions, solver, T_vars, R_vars, I_vars, rooms, instructors, timeslots):
        """Build solution DataFrame from solver values"""
        import pandas as pd
        print("   📊 Extracting solution ...")
        rows = []
        for i, s in enumerate(sessions):
            try:
                t = solver.Value(T_vars[i])
            except:
                t = None
            try:
                r_idx = solver.Value(R_vars[i])
            except:
                r_idx = None
            try:
                i_idx = solver.Value(I_vars[i])
            except:
                i_idx = None

            room_id = rooms[r_idx].get('RoomID', f"Room_{r_idx}") if (r_idx is not None and r_idx < len(rooms)) else f"Room_{r_idx}"
            instr = instructors[i_idx] if (i_idx is not None and i_idx < len(instructors)) else {}
            instr_id = instr.get('InstructorID', f"I_{i_idx}")
            instr_role = instr.get('Role', '')

            time_data = timeslots[t] if (t is not None and t < len(timeslots)) else {}
            time_slot_id = time_data.get('TimeSlotID', f"Slot_{t}")
            time_label = time_data.get('TimeSlotLabel', f"Slot_{t}")

            rows.append({
                'SessionID': s.get('SessionID'),
                'CourseID': s.get('CourseID'),
                'CourseName': s.get('CourseName', ''),
                'ClassType': s.get('ClassType', ''),
                'GroupID': s.get('GroupID'),
                'StudentCount': s.get('StudentCount'),
                'TimeSlot': t,
                'TimeSlotID': time_slot_id,
                'TimeSlotLabel': time_label,
                'RoomID': room_id,
                'InstructorID': instr_id,
                'InstructorRole': instr_role,
                'Level': s.get('Level'),
                'Specialization': s.get('Specialization'),
                'RequiredInstructorRole': s.get('RequiredInstructorRole', 'Professor')
            })
        df = pd.DataFrame(rows)
        print(f"   ✅ Extracted {len(df)} scheduled sessions")
        return df

    def _validate_global_solution_with_roles(self, solution_df, instructors):
        """Validate and print conflicts & role violations"""
        print("   🔍 Validating solution ...")
        if solution_df is None or len(solution_df) == 0:
            print("   ⚠️ No solution to validate.")
            return

        role_viol = solution_df[solution_df['InstructorRole'] != solution_df['RequiredInstructorRole']]
        if not role_viol.empty:
            print(f"   ❌ ROLE VIOLATIONS: {len(role_viol)} (showing up to 10):")
            for _, r in role_viol.head(10).iterrows():
                print(f"      {r['SessionID']}: {r['CourseID']} needs {r['RequiredInstructorRole']}, assigned {r['InstructorRole']}")
        else:
            print("   ✅ No role violations.")

        dup_room = solution_df.groupby(['RoomID', 'TimeSlot']).size()
        dup_room = dup_room[dup_room > 1]
        if len(dup_room) > 0:
            print(f"   ❌ ROOM-TIME CONFLICTS: {len(dup_room)}")
        else:
            print("   ✅ No room-time conflicts.")

        dup_instr = solution_df.groupby(['InstructorID', 'TimeSlot']).size()
        dup_instr = dup_instr[dup_instr > 1]
        if len(dup_instr) > 0:
            print(f"   ❌ INSTRUCTOR-TIME CONFLICTS: {len(dup_instr)}")
        else:
            print("   ✅ No instructor-time conflicts.")



In [ ]:
from ortools.sat.python import cp_model

model = cp_model.CpModel()
x = model.NewIntVar(0, 10, "x")
model.Maximize(x)

solver = cp_model.CpSolver()
solver.parameters.log_search_progress = True
solver.parameters.log_to_stdout = True

result = solver.Solve(model)
print("Finished:", result, "x =", solver.Value(x))


Finished: 4 x = 10


In [ ]:
def visualize_timetable_with_gaps(solution_df, level=None, highlight_gaps=True):
    """
    Professional timetable visualization compatible with GlobalTimetableSolver.

    Parameters
    ----------
    solution_df : pandas.DataFrame
        Output of GlobalTimetableSolver.solve_global()
    level : int or None
        If provided, visualize only this academic level.
    highlight_gaps : bool
        If True, explicitly mark empty slots as GAP.
    """

    print("\n📅 TIMETABLE VISUALIZATION")
    if level is not None:
        print(f"   🎓 Academic Level: {level}")
    print("=" * 90)

    # ---------- Filter ----------
    if level is not None:
        df = solution_df[solution_df["Level"] == level]
    else:
        df = solution_df.copy()

    if df.empty:
        print("⚠️ No sessions available for this selection.")
        return

    # ---------- Basic Sets ----------
    time_slots = sorted(df["TimeSlot"].unique())
    rooms = sorted(df["RoomID"].unique())

    print(f"📊 Summary:")
    print(f"   • Sessions : {len(df)}")
    print(f"   • TimeSlots: {len(time_slots)}")
    print(f"   • Rooms    : {len(rooms)}")
    print("-" * 90)

    # ---------- Build Grid ----------
    grid = {room: {t: None for t in time_slots} for room in rooms}

    for _, row in df.iterrows():
        grid[row["RoomID"]][row["TimeSlot"]] = row

    # ---------- Header ----------
    print(f"{'Room':<15}", end="")
    for t in time_slots:
        print(f"| T{t:^4} ", end="")
    print()
    print("-" * (15 + len(time_slots) * 8))

    # ---------- Rows ----------
    for room in rooms:
        print(f"{room:<15}", end="")
        for t in time_slots:
            session = grid[room][t]
            if session is not None:
                course = str(session["CourseID"])[:6]
                ctype = session["ClassType"][0]   # L / B
                role  = session["InstructorRole"][0]  # P / A
                print(f"| {course}-{ctype}{role:^2} ", end="")
            else:
                if highlight_gaps:
                    print(f"|  GAP   ", end="")
                else:
                    print(f"|        ", end="")
        print()

    # ---------- Utilization ----------
    total_slots = len(rooms) * len(time_slots)
    used_slots = len(df)
    gap_slots = total_slots - used_slots
    utilization = (used_slots / total_slots * 100) if total_slots else 0

    print("\n📈 UTILIZATION METRICS")
    print(f"   • Total slots : {total_slots}")
    print(f"   • Used slots  : {used_slots}")
    print(f"   • Gaps        : {gap_slots}")
    print(f"   • Utilization : {utilization:.1f}%")

    # ---------- Room Utilization ----------
    print("\n🏫 ROOM UTILIZATION (Top 10)")
    room_stats = []
    for room in rooms:
        used = sum(1 for t in time_slots if grid[room][t] is not None)
        util = used / len(time_slots) * 100 if time_slots else 0
        room_stats.append((room, used, util))

    room_stats.sort(key=lambda x: x[2], reverse=True)

    for room, used, util in room_stats[:10]:
        print(f"   • {room:<12}: {used}/{len(time_slots)} ({util:.1f}%)")

    if len(room_stats) > 10:
        print(f"   ... and {len(room_stats) - 10} more rooms")


In [ ]:
# ============================================================
# COMPLETE WORKFLOW — PHASED SOLVING (MATCHES SOLVER DESIGN)
# ============================================================

print("🎯 COMPLETE WORKFLOW WITH ROLE ENFORCEMENT (PHASED)")

# ------------------------------------------------------------
# 1) Create student groups
# ------------------------------------------------------------
print("🔄 CREATING GROUPS...")
improved_creator = ImprovedGroupCreator(normalized_data)
improved_lecture_groups, improved_lab_sections = improved_creator.create_all_groups()

# ------------------------------------------------------------
# 2) Generate sessions with role enforcement
# ------------------------------------------------------------
print("🔄 GENERATING SESSIONS WITH ROLE ENFORCEMENT...")
improved_session_generator = ImprovedSessionGenerator(
    normalized_data,
    improved_lecture_groups,
    improved_lab_sections
)
improved_sessions_df = improved_session_generator.generate_sessions()

# ------------------------------------------------------------
# 3) Initialize solver
# ------------------------------------------------------------
print("🔄 INITIALIZING SOLVER...")
global_solver = GlobalTimetableSolver(normalized_data)

# ------------------------------------------------------------
# 4) PHASE 1 — FEASIBILITY ONLY (MUST SUCCEED)
# ------------------------------------------------------------
print("\n🟢 PHASE 1: FEASIBILITY CHECK")
solution_phase1 = global_solver.solve_global(
    improved_sessions_df,
    time_limit_seconds=300,
    phase=1
)

if solution_phase1 is None:
    print("❌ PHASE 1 FAILED — MODEL IS INFEASIBLE")
    raise RuntimeError("Scheduling infeasible even without soft constraints")

# ------------------------------------------------------------
# 5) PHASE 2 — SAFE SOFT CONSTRAINTS
# ------------------------------------------------------------
print("\n🟡 PHASE 2: SAFE OPTIMIZATION")
solution_phase2 = global_solver.solve_global(
    improved_sessions_df,
    time_limit_seconds=300,
    phase=2
)

if solution_phase2 is None:
    print("⚠️ PHASE 2 FAILED — FALLING BACK TO PHASE 1 SOLUTION")
    final_solution = solution_phase1
else:
    final_solution = solution_phase2

# ------------------------------------------------------------
# 6) PHASE 3 — FULL OPTIMIZATION (OPTIONAL)
# ------------------------------------------------------------
print("\n🔵 PHASE 3: FULL OPTIMIZATION")
solution_phase3 = global_solver.solve_global(
    improved_sessions_df,
    time_limit_seconds=600,
    phase=3
)

if solution_phase3 is None:
    print("⚠️ PHASE 3 FAILED — USING BEST PREVIOUS SOLUTION")
else:
    final_solution = solution_phase3

# ------------------------------------------------------------
# 7) FINAL SOLUTION SUMMARY
# ------------------------------------------------------------
print("\n✅ SOLVER COMPLETE")
print("   • Sessions scheduled:", len(final_solution))
print("   • Levels:", sorted(final_solution["Level"].unique()))

# ------------------------------------------------------------
# 8) VISUALIZATION (MATCHES SOLVER OUTPUT)
# ------------------------------------------------------------
print("\n🔄 VISUALIZING FINAL TIMETABLE")

print("\n🏫 ROOM UTILIZATION VIEW")
visualize_timetable_with_gaps(final_solution)

print("\n👥 STUDENT GROUP GAP VIEW")
visualize_timetable_with_gaps(final_solution, highlight_gaps=True)

for level in sorted(final_solution["Level"].unique()):
    print(f"\n📘 LEVEL {level}")
    visualize_timetable_with_gaps(final_solution, level=level)

print("\n🎉 WORKFLOW COMPLETE — PHASED SOLVING SUCCESSFUL")


🎯 COMPLETE WORKFLOW WITH ROLE ENFORCEMENT (PHASED)
🔄 CREATING GROUPS...
👥 CREATING IMPROVED GROUPS (ONLY CORE FOR LEVELS WITH CORE COURSES)...
   • Processing levels: [np.int64(1), np.int64(2), np.int64(3), np.int64(4)]

   🎯 PROCESSING LEVEL 1:
      Total students: 225
      Core: 3 groups (Level has Core courses)
      Lab: 9 sections

   🎯 PROCESSING LEVEL 2:
      Total students: 225
      Core: 3 groups (Level has Core courses)
      Lab: 9 sections

   🎯 PROCESSING LEVEL 3:
      Total students: 205
      Core: 3 groups (Level has Core courses)
      AID: 1 groups
      CNC: 1 groups
      CSC: 1 groups
      BIF: 1 groups
      Lab: 9 sections

   🎯 PROCESSING LEVEL 4:
      Total students: 205
      Core: 0 groups (Level has NO Core courses)
      AID: 1 groups
      CNC: 1 groups
      CSC: 1 groups
      BIF: 1 groups
      Lab: 9 sections

🎯 IMPROVED GROUP CREATION COMPLETE!
📊 FINAL SUMMARY:
   • Lecture Groups: 17
   • Lab Sections: 36
   • Total Lecture Students: 1065
   

In [ ]:
# ============================================================
# FINAL EXPORT FOR DEPLOYMENT (STREAMLIT / DASHBOARD READY)
# ============================================================

import os
import json
import pickle
import shutil
import pandas as pd
from datetime import datetime
from google.colab import files

# ------------------------------------------------------------
# SAFETY CHECK
# ------------------------------------------------------------
assert "final_solution" in globals(), "❌ final_solution not found"
assert final_solution is not None, "❌ final_solution is None"

print("✅ Exporting final solution...")

# ------------------------------------------------------------
# CREATE OUTPUT DIRECTORY
# ------------------------------------------------------------
BASE_DIR = "deployment_data"
os.makedirs(BASE_DIR, exist_ok=True)

# ------------------------------------------------------------
# 1️⃣ MAIN SOLUTION (FAST LOAD FOR STREAMLIT)
# ------------------------------------------------------------
with open(f"{BASE_DIR}/solution_latest.pkl", "wb") as f:
    pickle.dump(final_solution, f)

final_solution.to_csv(
    f"{BASE_DIR}/solution_latest.csv",
    index=False
)

print("✔ Main solution saved")

# ------------------------------------------------------------
# 2️⃣ SOLUTION SPLIT BY LEVEL (VERY FAST FILTERING)
# ------------------------------------------------------------
solution_by_level = {
    int(level): df.reset_index(drop=True)
    for level, df in final_solution.groupby("Level")
}

with open(f"{BASE_DIR}/solution_by_level.pkl", "wb") as f:
    pickle.dump(solution_by_level, f)

print("✔ Per-level solutions saved")


# ------------------------------------------------------------
# 3️⃣ ROOM UTILIZATION (DASHBOARD STATS)
# ------------------------------------------------------------
room_util = (
    final_solution
    .groupby("RoomID")
    .size()
    .reset_index(name="Sessions")
    .sort_values("Sessions", ascending=False)
)

room_util["UtilizationPercent"] = (
    room_util["Sessions"] / final_solution["TimeSlot"].nunique() * 100
).round(1)

room_util.to_csv(
    f"{BASE_DIR}/room_utilization.csv",
    index=False
)

print("✔ Room utilization saved")

# ------------------------------------------------------------
# 4️⃣ INSTRUCTOR LOAD (WORKLOAD ANALYSIS)
# ------------------------------------------------------------
instructor_load = (
    final_solution
    .groupby(["InstructorID", "InstructorRole"])
    .size()
    .reset_index(name="Sessions")
    .sort_values("Sessions", ascending=False)
)

instructor_load.to_csv(
    f"{BASE_DIR}/instructor_load.csv",
    index=False
)

print("✔ Instructor load saved")

# ------------------------------------------------------------
# 5️⃣ GROUP GAP STATISTICS
# ------------------------------------------------------------
group_gaps = (
    final_solution
    .groupby("GroupID")["TimeSlot"]
    .apply(lambda x: x.sort_values().diff().gt(1).sum())
    .reset_index(name="GapCount")
    .sort_values("GapCount", ascending=False)
)

group_gaps.to_csv(
    f"{BASE_DIR}/group_gaps.csv",
    index=False
)

print("✔ Group gap statistics saved")

# ------------------------------------------------------------
# 6️⃣ METADATA (CRITICAL FOR VERSIONING)
# ------------------------------------------------------------
metadata = {
    "generated_at": datetime.utcnow().isoformat(),
    "total_sessions": int(len(final_solution)),
    "levels": sorted(map(int, final_solution["Level"].unique())),
    "timeslots": int(final_solution["TimeSlot"].nunique()),
    "rooms": int(final_solution["RoomID"].nunique()),
    "phases_used": [1, 2, 3],
    "solver": "OR-Tools CP-SAT",
    "institution": "CSIT",
    "status": "OPTIMAL_FEASIBLE"
}

with open(f"{BASE_DIR}/metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print("✔ Metadata saved")

# ------------------------------------------------------------
# 7️⃣ ZIP EVERYTHING
# ------------------------------------------------------------
ZIP_NAME = "CSIT_Timetable_Deployment_Files.zip"
shutil.make_archive(
    ZIP_NAME.replace(".zip", ""),
    "zip",
    BASE_DIR
)

# ------------------------------------------------------------
# 8️⃣ DOWNLOAD
# ------------------------------------------------------------
files.download(ZIP_NAME)

print("\n🎉 EXPORT COMPLETE — READY FOR DEPLOYMENT")


✅ Exporting final solution...
✔ Main solution saved
✔ Per-level solutions saved
✔ Room utilization saved
✔ Instructor load saved
✔ Group gap statistics saved
✔ Metadata saved


/tmp/ipython-input-2061212404.py:116: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "generated_at": datetime.utcnow().isoformat(),


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


🎉 EXPORT COMPLETE — READY FOR DEPLOYMENT


In [ ]:
import pandas as pd

solution = pd.read_pickle("deployment_data/solution_latest.pkl")
print(solution.head())
print(solution["Level"].unique())


  SessionID CourseID                   CourseName ClassType GroupID  \
0     S0001   CSC111  Fundamentals of Programming   Lecture    G001   
1     S0002   CSC111  Fundamentals of Programming   Lecture    G002   
2     S0003   CSC111  Fundamentals of Programming   Lecture    G003   
3     S0004   CSC111  Fundamentals of Programming       Lab    L001   
4     S0005   CSC111  Fundamentals of Programming       Lab    L002   

   StudentCount  TimeSlot  TimeSlotID              TimeSlotLabel     RoomID  \
0            75         8           9   Tuesday 9:00 AM-10:30 AM   B07 G.01   
1            75         2           3    Sunday 12:30 PM-2:00 PM   B08 G.41   
2            75         9          10  Tuesday 10:45 AM-12:15 PM   B09 G.35   
3            25         4           5    Monday 9:00 AM-10:30 AM  B07 F1.01   
4            25         5           6   Monday 10:45 AM-12:15 PM  B07 F1.02   

  InstructorID InstructorRole  Level Specialization RequiredInstructorRole  
0          P35      P

In [ ]:
levels = sorted(solution["Level"].dropna().unique())

for lvl in levels:
    lvl_df = solution[solution["Level"] == lvl]
    lvl_df.to_pickle(f"solution_level_{lvl}.pkl")
    lvl_df.to_csv(f"solution_level_{lvl}.csv", index=False)

print("✅ Per-level solutions created")


✅ Per-level solutions created


In [ ]:
# ============================================================
# 📦 DEPLOYMENT STEP: EXPORT DATA FOR STREAMLIT
# ============================================================
import os
import shutil

print("\n📦 STARTING DEPLOYMENT EXPORT...")

# 1. Create the output folder
deploy_dir = "deployment_data"
if not os.path.exists(deploy_dir):
    os.makedirs(deploy_dir)
    print(f"   📁 Created folder: {deploy_dir}")

# 2. Export the Main Solution (Critical for Streamlit)
if 'final_solution' in globals() and final_solution is not None:
    # Save as CSV (easiest for Streamlit to read)
    csv_path = os.path.join(deploy_dir, "solution_latest.csv")
    final_solution.to_csv(csv_path, index=False)
    print(f"   ✅ Saved Schedule: {csv_path}")

    # 3. Generate & Save Room Utilization Stats (Optional but good for Dashboard)
    #    We calculate this from the final_solution
    room_stats = final_solution.groupby("RoomID").size().reset_index(name="Sessions")

    # Calculate utilization % (Sessions / Total TimeSlots)
    total_slots = final_solution['TimeSlot'].nunique()
    room_stats['UtilizationPercent'] = (room_stats['Sessions'] / total_slots * 100).round(1)

    stats_path = os.path.join(deploy_dir, "room_utilization.csv")
    room_stats.to_csv(stats_path, index=False)
    print(f"   ✅ Saved Room Stats: {stats_path}")

else:
    print("   ❌ Error: 'final_solution' variable not found. Did the solver run successfully?")

print("🎉 DEPLOYMENT READY! You can now run 'streamlit run app.py'")


📦 STARTING DEPLOYMENT EXPORT...
   ✅ Saved Schedule: deployment_data/solution_latest.csv
   ✅ Saved Room Stats: deployment_data/room_utilization.csv
🎉 DEPLOYMENT READY! You can now run 'streamlit run app.py'


In [ ]:
# ============================================================
# 📦 DEPLOYMENT STEP: EXPORT DATA FOR STREAMLIT
# ============================================================
import os
import shutil

print("\n📦 STARTING DEPLOYMENT EXPORT...")

# 1. Create the output folder
deploy_dir = "deployment_data"
if not os.path.exists(deploy_dir):
    os.makedirs(deploy_dir)
    print(f"   📁 Created folder: {deploy_dir}")

# Check which variable exists (solution OR final_solution)
if 'solution' in globals():
    export_df = solution
elif 'final_solution' in globals():
    export_df = final_solution
else:
    export_df = None

# 2. Export
if export_df is not None and not export_df.empty:
    # Save as CSV (easiest for Streamlit to read)
    csv_path = os.path.join(deploy_dir, "solution_latest.csv")
    export_df.to_csv(csv_path, index=False)
    print(f"   ✅ Saved Schedule: {csv_path}")

    # 3. Generate & Save Room Utilization Stats
    room_stats = export_df.groupby("RoomID").size().reset_index(name="Sessions")

    # Calculate utilization % (Sessions / Total TimeSlots)
    if 'TimeSlot' in export_df.columns:
        total_slots = export_df['TimeSlot'].nunique()
        room_stats['UtilizationPercent'] = (room_stats['Sessions'] / total_slots * 100).round(1)

    stats_path = os.path.join(deploy_dir, "room_utilization.csv")
    room_stats.to_csv(stats_path, index=False)
    print(f"   ✅ Saved Room Stats: {stats_path}")

else:
    print("   ❌ Error: No schedule data found. Did the solver run successfully?")

print("🎉 DEPLOYMENT READY! You can now run 'streamlit run app.py'")



📦 STARTING DEPLOYMENT EXPORT...
   ✅ Saved Schedule: deployment_data/solution_latest.csv
   ✅ Saved Room Stats: deployment_data/room_utilization.csv
🎉 DEPLOYMENT READY! You can now run 'streamlit run app.py'
